# **Data Science Project 4**

# **Movie Review Sentiment Analysis**
## ***This AI project cleans text and automatically classifies movie reviews as positive or negative.***

## **Import Libraries**

In [ ]:
import re
import nltk
import numpy as np
from nltk.corpus import movie_reviews, stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import classification_report, accuracy_score
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('movie_reviews')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


True

## **Load raw text data from NLTK movie_reviews**

In [ ]:
def load_dataset():
    docs, labels = [], []
    for category in movie_reviews.categories():
        for fileid in movie_reviews.fileids(category):
            raw_text = movie_reviews.raw(fileid)
            docs.append(raw_text)
            labels.append(1 if category == "pos" else 0)
    return docs, labels

## **Clean tokens, remove stop-words,lemmatize with POS-guided WordNetLemmatizer**

In [ ]:
_BASE_STOPS = set(stopwords.words("english"))
_NEGATIONS  = {"no", "not", "nor", "never", "neither",
               "wasn't", "isn't", "aren't", "weren't",
               "don't", "doesn't", "didn't", "won't",
               "wouldn't", "couldn't", "shouldn't", "can't",
               "cannot", "hardly", "barely", "scarcely"}
STOP_WORDS = _BASE_STOPS - _NEGATIONS  # set-based union operation

lemmatizer = WordNetLemmatizer()


def _treebank_to_wordnet(treebank_tag):
    """Map Penn Treebank POS tags → WordNet POS constants."""
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    elif treebank_tag.startswith("V"):
        return wordnet.VERB
    elif treebank_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default


def preprocess(text):
    """
    Full pre-processing pipeline:
      1. Character normalization  (strip HTML, punctuation, digits)
      2. Lowercasing
      3. Tokenization
      4. Stop-word removal        (negations preserved)
      5. POS-guided Lemmatization (WordNetLemmatizer)
    """
    # Character normalization
    text = re.sub(r"<[^>]+>", " ", text)          # remove HTML tags
    text = re.sub(r"[^a-zA-Z\s]", " ", text)      # keep only letters
    text = text.lower().strip()

    # Tokenize
    tokens = word_tokenize(text)

    # Stop-word removal (negations kept)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1]

    # POS-guided lemmatization — mandatory per project blueprint
    pos_tags = pos_tag(tokens)
    tokens = [
        lemmatizer.lemmatize(word, pos=_treebank_to_wordnet(tag))
        for word, tag in pos_tags
    ]

    return " ".join(tokens)

## **TF-IDF with Unigrams + Bigrams,stored as SciPy CSR sparse matrix**

In [ ]:
def build_vectorizer():
    """
    TF-IDF with bigrams to capture negated phrases like "not good".
    max_features=10000 prevents dimensionality explosion.
    min_df=2 drops rare typos/one-off tokens.
    Output is a SciPy CSR sparse matrix (memory-efficient).
    """
    return TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=10_000,
        min_df=2,
        sublinear_tf=True   # apply log(1+tf) — standard for NLP
    )

## **Naive Bayes classifier with Laplace smoothing**

In [ ]:
def build_classifier(labels):
    """
    Use ComplementNB if dataset is imbalanced, MultinomialNB if balanced.
    Both use Laplace smoothing (alpha=1.0) to prevent zero-frequency crash.
    """
    pos_count = sum(labels)
    neg_count = len(labels) - pos_count
    ratio = max(pos_count, neg_count) / min(pos_count, neg_count)

    if ratio > 1.5:
        print(f"  Imbalanced dataset detected (ratio={ratio:.2f}) → ComplementNB")
        return ComplementNB(alpha=1.0)
    else:
        print(f"  Balanced dataset detected (ratio={ratio:.2f}) → MultinomialNB")
        return MultinomialNB(alpha=1.0)


## **Predict sentiment on new reviews**

In [ ]:
def predict_sentiment(texts, vectorizer, model):
    """Transform new raw text and return Positive/Negative predictions."""
    cleaned = [preprocess(t) for t in texts]
    X = vectorizer.transform(cleaned)       # CSR sparse matrix
    preds = model.predict(X)
    probs = model.predict_proba(X)
    return preds, probs

## **Assemble and run the full pipeline**

In [ ]:
def main():
    print("=" * 60)
    print("  Project 4: NLP & Sentiment Analysis")
    print("  DecodeLabs | Batch 2026")
    print("=" * 60)

    # --- Ingest ---
    print("\n[1/5] Loading dataset...")
    docs, labels = load_dataset()
    print(f"  Total samples : {len(docs)}")
    print(f"  Positive      : {sum(labels)}")
    print(f"  Negative      : {len(labels) - sum(labels)}")

    # --- Pre-process ---
    print("\n[2/5] Pre-processing text (this may take a moment)...")
    cleaned_docs = [preprocess(doc) for doc in docs]
    print("  Sample cleaned review (first 120 chars):")
    print(f"  {cleaned_docs[0][:120]}...")

    # --- Train/Test Split ---
    X_train, X_test, y_train, y_test = train_test_split(
        cleaned_docs, labels,
        test_size=0.2,
        random_state=42,
        stratify=labels
    )
    print(f"\n  Train size : {len(X_train)} | Test size : {len(X_test)}")

    # --- Vectorize ---
    print("\n[3/5] Vectorizing with TF-IDF (Unigrams + Bigrams)...")
    vectorizer = build_vectorizer()
    X_train_tfidf = vectorizer.fit_transform(X_train)   # CSR sparse matrix
    X_test_tfidf  = vectorizer.transform(X_test)
    print(f"  Vocabulary size   : {len(vectorizer.vocabulary_)}")
    print(f"  Feature matrix    : {X_train_tfidf.shape[0]} x {X_train_tfidf.shape[1]} (CSR sparse)")

    # --- Model ---
    print("\n[4/5] Selecting and training classifier...")
    model = build_classifier(labels)
    model.fit(X_train_tfidf, y_train)
    print("  Training complete.")

    # --- Evaluate ---
    print("\n[5/5] Evaluating on held-out test set...")
    y_pred = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, y_pred)
    print(f"\n  Accuracy : {acc * 100:.2f}%\n")
    print(classification_report(
        y_test, y_pred,
        target_names=["Negative", "Positive"]
    ))

    # --- Live Inference Demo ---
    print("=" * 60)
    print("  LIVE INFERENCE DEMO")
    print("=" * 60)
    sample_reviews = [
        "This movie was absolutely fantastic! I loved every moment.",
        "Terrible film. I wasted two hours of my life. Not enjoyable at all.",
        "It was not bad, actually quite enjoyable in parts.",
        "Boring, predictable, and the acting was not convincing whatsoever.",
    ]
    preds, probs = predict_sentiment(sample_reviews, vectorizer, model)
    label_map = {0: "NEGATIVE", 1: "POSITIVE"}

    for review, pred, prob in zip(sample_reviews, preds, probs):
        confidence = np.max(prob) * 100
        print(f"\n  Review     : {review}")
        print(f"  Prediction : {label_map[pred]}  (confidence: {confidence:.1f}%)")

    print("\n" + "=" * 60)
    print("  Pipeline complete.")
    print("=" * 60)


if __name__ == "__main__":
    main()


  Project 4: NLP & Sentiment Analysis
  DecodeLabs | Batch 2026

[1/5] Loading dataset...
  Total samples : 2000
  Positive      : 1000
  Negative      : 1000

[2/5] Pre-processing text (this may take a moment)...
  Sample cleaned review (first 120 chars):
  plot two teen couple go church party drink drive get accident one guy die girlfriend continue see life nightmare deal wa...

  Train size : 1600 | Test size : 400

[3/5] Vectorizing with TF-IDF (Unigrams + Bigrams)...
  Vocabulary size   : 10000
  Feature matrix    : 1600 x 10000 (CSR sparse)

[4/5] Selecting and training classifier...
  Balanced dataset detected (ratio=1.00) → MultinomialNB
  Training complete.

[5/5] Evaluating on held-out test set...

  Accuracy : 84.00%

              precision    recall  f1-score   support

    Negative       0.84      0.83      0.84       200
    Positive       0.84      0.84      0.84       200

    accuracy                           0.84       400
   macro avg       0.84      0.84      0.84